# 03 · 부분 배치 스케줄링과 버퍼 수명

이 노트북은 [GzDRL 논문](https://arxiv.org/html/2609.13243v1)의 Algorithm 1, 식 (3), 실험 V-C의 **교육용 toy scheduler**다. 실제 Gazebo, ROS 2, PPO, C++ GazeboPool, CPU pinning, zero-copy C++/NumPy 경계 또는 로봇 시뮬레이션을 재현하지 않는다. 여기서 나온 처리량은 toy Python 함수의 속도일 뿐 논문의 환경 스텝/초와 비교할 수 없다. 외부 패키지는 NumPy만 사용한다.

논문의 $N_E$는 환경 수, $N_T$는 worker 수, $N_B\leq N_E$는 수신할 부분 배치 크기다. worker가 환경별 작업을 동적으로 받아 관측·보상·종료를 기록하고, 학습자는 $N_B$개가 완료되면 `env_id`와 관측을 함께 받아 해당 환경에 다음 동작을 보낸다. 여기서는 `Run(1)` 대신 결정적인 숫자 한 번 갱신을 사용하며, worker 완료 시각은 실제 동시 실행이 아닌 가상 시간이다.


In [ ]:
import time
import numpy as np

N_E, N_T, N_B = 8, 3, 2
initial_states = 0.1 * np.arange(N_E, dtype=float)
actions = 0.05 * np.cos(np.arange(N_E, dtype=float))

def virtual_round(states, requested_actions, n_workers, batch_size):
    states = np.asarray(states, dtype=float)
    requested_actions = np.asarray(requested_actions, dtype=float)
    n_env = len(states)
    if len(requested_actions) != n_env or not (1 <= n_workers <= n_env):
        raise ValueError('환경·동작·worker 수를 확인하세요.')
    if not (1 <= batch_size <= n_env) or n_env % batch_size:
        raise ValueError('이 단일 라운드 toy에서는 batch_size가 N_E를 나누어야 합니다.')
    worker_free_at = np.zeros(n_workers, dtype=float)
    completions = []
    for env_id, requested in enumerate(requested_actions):
        worker = int(np.argmin(worker_free_at))
        action = float(np.clip(requested, -1.0, 1.0))
        next_state = states[env_id] + action  # Gazebo 물리 전이가 아닌 숫자 toy
        reward = -(next_state ** 2)
        done = float(abs(next_state) > 1.0)
        work_cost = 1.0 + 0.1 * (env_id % 4)  # 가상 완료 시각만 결정
        worker_free_at[worker] += work_cost
        completions.append((worker_free_at[worker], env_id, next_state, reward, done))
    completions.sort(key=lambda row: (row[0], row[1]))
    owner_buffer = np.asarray([row[1:] for row in completions], dtype=float)
    completion_times = np.asarray([row[0] for row in completions])
    batches = [owner_buffer[i:i + batch_size] for i in range(0, n_env, batch_size)]
    receive_times = completion_times[batch_size - 1::batch_size]
    return owner_buffer, batches, receive_times

owner, batches, partial_recv = virtual_round(initial_states, actions, N_T, N_B)
_, full_batches, full_recv = virtual_round(initial_states, actions, N_T, N_E)
env_ids = owner[:, 0].astype(int)
assert len(batches) == N_E // N_B
assert all(len(batch) == N_B for batch in batches)
assert np.array_equal(np.sort(env_ids), np.arange(N_E))
assert np.allclose(owner[:, 1], initial_states[env_ids] + actions[env_ids])
assert np.allclose(owner[:, 2], -(owner[:, 1] ** 2))
assert np.shares_memory(batches[0], owner)
assert partial_recv[0] < full_recv[0]
assert partial_recv[-1] == full_recv[-1]
print('env_id 누락/중복 없음; 부분 배치 첫 수신 가상 시각:', partial_recv[0])
print('전체 배치 첫 수신 가상 시각:', full_recv[0])


## zero-copy 관측 뷰와 오염 방지

논문은 C++ 소유 회전 버퍼에 Python이 복사 없는 NumPy view로 접근한다. 다음은 **Python 메모리 안에서만** 같은 버퍼-수명 원칙을 실습한다. view를 소비하는 동안 해당 버퍼를 재사용하면 관측이 오염되므로, 대여 중인 버퍼는 다시 획득할 수 없게 한다. 오래 보관할 데이터는 `.copy()`해야 한다. 이것은 실제 C++ 경계 검증이 아니다.


In [ ]:
class TwoBufferPool:
    def __init__(self, shape):
        self.buffers = [np.empty(shape, dtype=float) for _ in range(2)]
        self.in_use = [False, False]

    def acquire(self):
        for index, busy in enumerate(self.in_use):
            if not busy:
                self.in_use[index] = True
                return index, self.buffers[index]
        raise RuntimeError('모든 관측 버퍼가 대여 중입니다.')

    def release(self, index):
        if not self.in_use[index]:
            raise RuntimeError('대여하지 않은 버퍼는 반납할 수 없습니다.')
        self.in_use[index] = False

pool = TwoBufferPool((N_B, 4))
i0, view0 = pool.acquire()
view0[:] = batches[0]
snapshot0 = view0.copy()
i1, view1 = pool.acquire()
view1[:] = batches[1]
snapshot1 = view1.copy()
assert np.array_equal(view0, snapshot0)  # 다른 버퍼를 채워도 첫 view는 유지
assert np.shares_memory(view0, pool.buffers[i0])
assert np.shares_memory(view1, pool.buffers[i1])
try:
    pool.acquire()
except RuntimeError:
    pass
else:
    raise AssertionError('대여 중인 버퍼를 덮어쓰면 안 됩니다.')
pool.release(i0)
i2, view2 = pool.acquire()
assert i2 == i0
view2[:] = batches[2]
assert np.array_equal(view1, snapshot1)  # 두 번째 대여는 여전히 안전
assert np.array_equal(snapshot0, batches[0])  # 복사본은 재사용 후에도 불변
pool.release(i1)
pool.release(i2)
print('대여 중인 버퍼 재사용 금지와 명시적 복사 보존 확인')


## V-C 방식으로 통계 읽기

논문 V-C는 $N_E\in\{1,4,8,16,32,64,128\}$, $N_B\approx N_E/3$, $N_T=\min(N_E,N_{\mathrm{CPU}})$로 구성하고, 설정마다 1,000스텝 준비 실행 후 10,000 환경 전이를 5회 측정해 초당 완료 환경 전이와 95% Student $t$ 신뢰구간을 보고한다. 아래는 시간을 줄이기 위해 100회 준비·1,000회 측정하는 **Python toy 함수 벤치마크**다. 가상 worker 시각은 실제 병렬 실행을 뜻하지 않는다. 부분 배치가 언제나 더 빠르다는 주장을 검증하지 않으며, 환경 성능·운영체제 부하에 따라 결과가 달라진다.


In [ ]:
def measure_toy_rate(batch_size, warmup=100, measured=1000):
    for _ in range(warmup):
        virtual_round(initial_states, actions, N_T, batch_size)
    started = time.perf_counter()
    for _ in range(measured):
        virtual_round(initial_states, actions, N_T, batch_size)
    elapsed = time.perf_counter() - started
    return N_E * measured / elapsed

def summarize_five_trials(batch_size):
    rates = np.asarray([measure_toy_rate(batch_size) for _ in range(5)])
    # 자유도 4의 양측 95% Student t 임계값. SciPy 의존성을 피한다.
    t_975_df4 = 2.7764451051977987
    mean = float(np.mean(rates))
    half_width = float(t_975_df4 * np.std(rates, ddof=1) / np.sqrt(5))
    assert rates.shape == (5,)
    assert np.all(np.isfinite(rates)) and np.all(rates > 0)
    assert np.isfinite(half_width) and half_width >= 0
    return rates, mean, half_width

partial_rates, partial_mean, partial_ci = summarize_five_trials(N_B)
full_rates, full_mean, full_ci = summarize_five_trials(N_E)
print(f'부분 배치 Python toy: {partial_mean:.0f} ± {partial_ci:.0f} 전이/s (5회, 95% CI)')
print(f'전체 배치 Python toy: {full_mean:.0f} ± {full_ci:.0f} 전이/s (5회, 95% CI)')
print('위 수치는 GazeboPool/ROS 2/타 시뮬레이터의 논문 측정값이 아닙니다.')
